[EKF localization](/aiml-common/lectures/rse/ekf-localization/index) assumes that we know the positions of the landmarks. Suppose instead that we place a TurtleBot in a room for which we have no map. When its sensor detects a pillar, how do we assign world coordinates to that pillar? We need the pose of the robot. But without a map, the robot cannot use the pillar to determine its pose.

In simultaneous localization and mapping (SLAM), we estimate the pose and the map together. The distinction from localization is that landmark positions are now unknown quantities in the estimation problem.

Let $a_{1:t}$ denote the actions and $z_{1:t}$ the measurements collected up to time $t$. We distinguish two estimation problems. In *full SLAM*, we estimate the entire trajectory $s_{1:t}$ and the map $m$:

$$
p(s_{1:t},m\mid z_{1:t},a_{1:t}).
$$

In *online SLAM*, we retain the current pose and the map:

$$
p(s_t,m\mid z_{1:t},a_{1:t}).
$$

EKF SLAM is an online method. We will study it for a planar robot and a map of point landmarks, using the formulation in Chapter 10 of Thrun, Burgard and Fox [@Thrun2005-dx]. The [implementation](/aiml-common/lectures/rse/ekf-slam/index) in the next section considers two sensors, a LIDAR and a D435i depth camera, which supply landmark measurements to the same estimator.

In [ ]:
%config InlineBackend.figure_formats = ["svg"]

import matplotlib.pyplot as plt
import numpy as np
from cycler import cycler

plt.rcParams.update(
    {
        "figure.figsize": (6.5, 3.5),
        "font.size": 9,
        "axes.titlesize": 10,
        "axes.labelsize": 9,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 8,
        "axes.prop_cycle": cycler(
            color=["#0072B2", "#D55E00", "#009E73", "#cb77a6", "#c28600", "#1d99e0", "#7f7f7f"]
        ),
        "axes.grid": True,
        "grid.alpha": 0.3,
        "svg.fonttype": "path",
    }
)
BLUE, VERMILLION, GREEN, PINK, GOLD, SKY, GRAY = (
    "#0072B2", "#D55E00", "#009E73", "#cb77a6", "#c28600", "#1d99e0", "#7f7f7f",
)


def ellipse(mean, cov, k=2.0, n=100):
    """Points on the k-sigma ellipse of a 2D Gaussian."""
    vals, vecs = np.linalg.eigh(cov)
    t = np.linspace(0, 2 * np.pi, n)
    return mean[:, None] + vecs @ (k * np.sqrt(np.maximum(vals, 0))[:, None] * np.vstack([np.cos(t), np.sin(t)]))

## A robot and a landmark on a line

Before considering the planar robot, let us examine how a single range measurement affects two uncertain positions. The robot is at an estimated position of 1 m, with a standard deviation of 20 cm. A landmark is estimated to be at 4 m, with a standard deviation of 10 cm. Assume that the two estimates are initially independent.

We would therefore expect the sensor to report a distance of 3 m. Suppose it reports 3.1 m instead. We could account for the extra 10 cm by moving the estimated landmark position to 4.1 m, by moving the estimated robot position to 0.9 m, or by changing both estimates. The Kalman filter must decide how to distribute the correction, taking into account the uncertainty in each position and in the measurement.

In [ ]:
from scipy.stats import norm

fig, ax = plt.subplots(figsize=(6.5, 3.0))
p = np.linspace(0, 5, 1000)
ax.fill_between(p, norm.pdf(p, 1.0, 0.2), color=BLUE, alpha=0.25, lw=0)
ax.plot(p, norm.pdf(p, 1.0, 0.2), color=BLUE, lw=1.5, label=r"robot $x$: mean 1 m, $\sigma$ = 0.2 m")
ax.fill_between(p, norm.pdf(p, 4.0, 0.1), color=VERMILLION, alpha=0.25, lw=0)
ax.plot(p, norm.pdf(p, 4.0, 0.1), color=VERMILLION, lw=1.5, label=r"landmark $m$: mean 4 m, $\sigma$ = 0.1 m")


def span(x0, x1, y, text, color, ls="-"):
    ax.annotate("", xy=(x1, y), xytext=(x0, y),
                arrowprops={"arrowstyle": "<->", "color": color, "lw": 1.3, "ls": ls, "shrinkA": 0, "shrinkB": 0})
    ax.text((x0 + x1) / 2, y + 0.12, text, ha="center", va="bottom", fontsize=8, color=color)


span(1.0, 4.0, 4.6, "predicted range 3.0 m", "black")
span(1.0, 4.1, 5.5, "measured 3.1 m: landmark at 4.1 m?", VERMILLION, ls="--")
span(0.9, 4.0, 6.4, "measured 3.1 m: robot at 0.9 m?", BLUE, ls="--")
ax.set_xlim(0, 5)
ax.set_ylim(0, 7.3)
ax.set_yticks([])
ax.grid(False)
ax.set_xlabel("position on the line (m)")
ax.set_title("physical space: a robot and a landmark on a line")
ax.legend(loc="center left", fontsize=7)
fig.tight_layout()
plt.show()

The figure shows the two prior probability densities on the same physical line. The dashed arrows illustrate two possible explanations for the measurement; neither is necessarily the actual configuration.

Since the landmark is ahead of the robot, we model the measurement as

$$
z=m-x+\epsilon,\qquad \epsilon\sim\mathcal N(0,R),
$$

where $x$ and $m$ denote the robot and landmark positions, respectively. More precisely, we use a signed distance model, which agrees with range when $m>x$. Here the measurement standard deviation is 0.1 m, so $R=0.01\,\mathrm{m}^2$.

Let us apply the Kalman filter from the earlier sections, with the state ordered as $(x,m)^\top$:

$$
\bar\mu=\begin{bmatrix}1\\4\end{bmatrix},\qquad
\bar\Sigma=\begin{bmatrix}0.04&0\\0&0.01\end{bmatrix}.
$$

The measurement function is linear, with $H=[-1\;\;1]$. The innovation is $\nu=3.1-3.0=0.1$ m. No EKF linearization is needed in this example.

In [ ]:
mu_1d = np.array([1.0, 4.0])  # robot x, landmark x (m)
Sigma_1d = np.diag([0.04, 0.01])
H_1d = np.array([[-1.0, 1.0]])  # z = m - x
R_1d = 0.01
nu_1d = np.array([3.1]) - H_1d @ mu_1d
S_1d = H_1d @ Sigma_1d @ H_1d.T + R_1d
K_1d = Sigma_1d @ H_1d.T @ np.linalg.inv(S_1d)
mu_1d_post = mu_1d + K_1d @ nu_1d
Sigma_1d_post = (np.eye(2) - K_1d @ H_1d) @ Sigma_1d
print("innovation variance S =", S_1d.round(3).item())
print("gain K =", K_1d.ravel().round(3))
print("posterior mean (x, m) =", mu_1d_post.round(3))
print("posterior covariance =\n", Sigma_1d_post.round(4))
print("correlation of x and m =", round(Sigma_1d_post[0, 1] / np.sqrt(Sigma_1d_post[0, 0] * Sigma_1d_post[1, 1]), 2))


The posterior mean and covariance are

$$
\mu=\begin{bmatrix}0.9333\\4.0167\end{bmatrix},\qquad
\Sigma=\begin{bmatrix}0.0133&0.0067\\0.0067&0.0083\end{bmatrix}.
$$

The robot position estimate decreases by 6.7 cm and the landmark position estimate increases by 1.7 cm. The robot receives the larger correction because its prior variance is four times that of the landmark. The combined correction is less than 10 cm because the measurement itself is uncertain.

## Correlation from a single update

Notice the change in the covariance matrix. Its off-diagonal entries, which were initially zero, are now positive. To interpret these entries, we need a different plot from the previous one. Put the robot position $x$ on the horizontal axis and the landmark position $m$ on the vertical axis. A point in this plot specifies an entire configuration of our one-dimensional world.

If the sensor were exact, all configurations consistent with the measurement would satisfy

$$
m=x+3.1.
$$

For instance, $(0.9,4.0)$ and $(1.0,4.1)$ both satisfy this equation. We cannot tell them apart using range alone. With a noisy sensor, configurations near the line are more likely than those far from it.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.6))
xs = np.linspace(0.4, 1.6, 2)
ax.fill_between(xs, xs + 3.1 - 2 * np.sqrt(R_1d), xs + 3.1 + 2 * np.sqrt(R_1d), color=GRAY, alpha=0.2, lw=0,
                label=r"measurement band ($\pm 2\sqrt{R}$)")
ax.plot(xs, xs + 3.1, color=GRAY, lw=1)
e = ellipse(mu_1d, Sigma_1d)
ax.plot(e[0], e[1], color=BLUE, ls="--", lw=1.5, label=r"prior ($2\sigma$ contour)")
ax.plot(*mu_1d, "o", color=BLUE, ms=6, mfc="white", label="prior mean")
e = ellipse(mu_1d_post, Sigma_1d_post)
ax.plot(e[0], e[1], color=VERMILLION, lw=2, label=r"posterior ($2\sigma$ contour)")
ax.plot(*mu_1d_post, "o", color=VERMILLION, ms=6, label="posterior mean")
ax.annotate("", xy=mu_1d_post, xytext=mu_1d, arrowprops={"arrowstyle": "->", "color": "black", "lw": 1})
ax.set_aspect("equal")
ax.set_xlim(0.5, 1.5)
ax.set_ylim(3.65, 4.35)
ax.set_xlabel("robot position $x$ (m)")
ax.set_ylabel("landmark position $m$ (m)")
ax.set_title("joint state space (not physical space)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2)
fig.tight_layout()
plt.show()

The blue ellipse describes the prior joint distribution, and the orange ellipse describes the posterior. Both are contours at Mahalanobis distance 2, enclosing about 86.5% of the probability of their respective two-dimensional Gaussians. The gray band indicates configurations within two measurement standard deviations of the observed range; it is not a boundary beyond which configurations are impossible.

The orange ellipse is elongated in a direction approximately parallel to the measurement line. If the robot is farther to the right than we estimate, the landmark is likely to be farther to the right as well. This is what the positive covariance $\Sigma_{xm}=0.006667\,\mathrm{m}^2$ tells us. The correlation coefficient is about 0.63.

We can also explain the result directly from Bayes' rule. The independent prior factors into $p(x)p(m)$, so

$$
p(x,m\mid z)\propto p(z\mid x,m)p(x)p(m).
$$

The likelihood depends on the difference $m-x$. Multiplying by this likelihood couples the two variables, and in this example the posterior no longer factors into separate robot and landmark distributions.

Suppose the measured range had been 3.0 m instead of 3.1 m. The innovation would have been zero and the two means would have stayed where they were. Nevertheless, the covariance matrix would have changed by exactly the same amount: in this linear model, the covariance update depends on $H$, $R$ and the prior covariance, not on the value of $z$. The measurement creates correlation even when there is no correction to the mean.

### What did the measurement tell us about the map?

The quantity measured by the sensor was the relative position $m-x$. Before the update its variance was

$$
\operatorname{Var}(m-x)=0.04+0.01=0.05\,\mathrm{m}^2.
$$

After the update we must account for the cross-covariance:

$$
\operatorname{Var}(m-x)=\Sigma_{mm}+\Sigma_{xx}-2\Sigma_{xm}
=0.00833\,\mathrm{m}^2.
$$

The standard deviation of the relative position has gone from 22.4 cm to 9.1 cm. Compare this with the uncertainty in the individual positions:

| Quantity | Prior standard deviation | Posterior standard deviation |
| --- | ---: | ---: |
| Robot position | 20.0 cm | 11.5 cm |
| Landmark position | 10.0 cm | 9.1 cm |
| Relative position | 22.4 cm | 9.1 cm |

<Accordion title="Where the variance of the relative position comes from">

Write each position as its mean plus an error, $x = \mu_x + e_x$ and $m = \mu_m + e_m$. The error of $m - x$ is then $e_m - e_x$, and its variance is the expected square of that error:

$$
\begin{aligned}
\operatorname{Var}(m-x) &= \mathbb E\big[(e_m - e_x)^2\big] \\
&= \mathbb E[e_m^2] + \mathbb E[e_x^2] - 2\,\mathbb E[e_m e_x] \\
&= \Sigma_{mm} + \Sigma_{xx} - 2\Sigma_{xm}.
\end{aligned}
$$

The last term is the cross-covariance, the off-diagonal entry of $\Sigma$. Before the update the two estimates are independent, so $\Sigma_{xm} = 0$ and the two variances simply add: $0.01 + 0.04 = 0.05\ \mathrm{m}^2$, a standard deviation of 22.4 cm.

After the update, with $\Sigma_{xx} = 0.0133$, $\Sigma_{mm} = 0.0083$ and $\Sigma_{xm} = 0.0067$,

$$
\operatorname{Var}(m-x) = 0.0083 + 0.0133 - 2(0.0067) = 0.0083\ \mathrm{m}^2,
$$

a standard deviation of 9.1 cm. The positive covariance enters with a minus sign. It means the errors $e_x$ and $e_m$ tend to have the same sign: if the robot is estimated too far to the right, the landmark probably is too. In the difference $e_m - e_x$ the two errors largely cancel, so the gap between robot and landmark is known much better than either position.

The same number follows without $\Sigma$. Before the update the relative position is known with variance 0.05. The sensor measures that same quantity with variance $R = 0.01$. Combining two independent estimates of one quantity adds their inverse variances:

$$
\operatorname{Var}(m-x) = \left(\frac{1}{0.05} + \frac{1}{0.01}\right)^{-1} = \frac{1}{20 + 100} = 0.0083\ \mathrm{m}^2.
$$

This is the one-dimensional Kalman update applied to $m - x$ directly. The result, 9.1 cm, is slightly better than the sensor's 10 cm, because the prior adds a little information.

In matrix form, $\operatorname{Var}(m-x) = w^\top \Sigma w$ with $w = (-1, 1)^\top$, which is $H \Sigma H^\top$. For the prior, $H \bar\Sigma H^\top = 0.05$ is the term that appears in the innovation variance $S = H \bar\Sigma H^\top + R = 0.06$.

</Accordion>

This distinction will matter when we have many landmarks. We may know the distances between landmarks quite accurately even when their coordinates in the world frame remain uncertain.

## The EKF SLAM state

We now return to a planar robot. Let $s_t=(x_t,y_t,\theta_t)^\top$ be its pose and let $m_k=(m_{k,x},m_{k,y})^\top$ be the coordinates of landmark $k$. With $N$ landmarks in the map, the state is

$$
y_t=\begin{bmatrix}s_t\\m\end{bmatrix}
=(x_t,y_t,\theta_t,m_{1,x},m_{1,y},\ldots,m_{N,x},m_{N,y})^\top.
$$

As in EKF localization, the filter approximates its belief by a Gaussian, $\mathtt{bel}(y_t)=\mathcal N(\mu_t,\Sigma_t)$. The difference is that the Gaussian now describes a $(3+2N)$-dimensional state. Thrun, Burgard and Fox also store a signature for each landmark (equation 10.7); we omit that component because the pillars in our example are indistinguishable by appearance.

It is useful to partition the covariance according to the pose and map:

$$
\Sigma_t=\begin{bmatrix}
\Sigma_{ss}&\Sigma_{sm}\\
\Sigma_{ms}&\Sigma_{mm}
\end{bmatrix}.
$$

The $3\times3$ block $\Sigma_{ss}$ describes pose uncertainty. The $2N\times2N$ block $\Sigma_{mm}$ includes the uncertainty of individual landmarks as well as their mutual cross-covariances. The rectangular block $\Sigma_{sm}$ describes how the pose estimate is correlated with each of the landmark estimates. By symmetry, $\Sigma_{ms}=\Sigma_{sm}^\top$.

The small example above explains why we cannot simply discard the off-diagonal blocks. They describe dependencies produced by past measurements. In a SLAM map, these dependencies also arise when several landmark positions are inferred from an uncertain pose.

## Prediction

Suppose the robot executes an action $a_t$. We already know from EKF localization how to predict the new pose using the motion model $g$. What should happen to the map? The landmarks are stationary, so their estimated coordinates do not change during prediction.

$$
\bar\mu_t=\begin{bmatrix}g(a_t,\mu_{t-1,s})\\\mu_{t-1,m}\end{bmatrix}.
$$

Let $G_t$ and $V_t$ be the Jacobians of the motion model with respect to the pose and the wheel speeds, respectively, and let $M_t$ be the wheel-speed noise covariance. The pose process noise is $Q_t=V_tM_tV_t^\top$. Assuming independent process noise and a static map, the augmented prediction is

$$
\bar\Sigma_t=\begin{bmatrix}G_t&0\\0&I\end{bmatrix}
\Sigma_{t-1}\begin{bmatrix}G_t&0\\0&I\end{bmatrix}^\top
+\begin{bmatrix}Q_t&0\\0&0\end{bmatrix}.
$$

Multiplying out the blocks gives

$$
\begin{aligned}
\bar\Sigma_{ss}&=G_t\Sigma_{ss}G_t^\top+Q_t,\\
\bar\Sigma_{sm}&=G_t\Sigma_{sm},\\
\bar\Sigma_{mm}&=\Sigma_{mm}.
\end{aligned}
$$

Here the blocks on the right belong to time $t-1$. The map's marginal distribution remains unchanged, but its relationship to the robot's new pose generally changes. This is why propagating only the $3\times3$ pose covariance is insufficient.

## Measurements of known landmarks

Suppose the robot detects landmark $k$ and that we know which landmark produced the measurement. As in localization, the measurement consists of a range and bearing:

$$
h(s,m_k)=\begin{bmatrix}
\sqrt{(m_{k,x}-x)^2+(m_{k,y}-y)^2}\\
\operatorname{atan2}(m_{k,y}-y,m_{k,x}-x)-\theta
\end{bmatrix}.
$$

In localization, the landmark coordinates were constants when we differentiated $h$. Now both $s$ and $m_k$ belong to the state. The full measurement Jacobian has two potentially nonzero blocks:

$$
H_t^k=\begin{bmatrix}\dfrac{\partial h}{\partial s}&0&\cdots&\dfrac{\partial h}{\partial m_k}&\cdots&0\end{bmatrix}.
$$

It is evaluated at the predicted mean. Columns belonging to other landmarks are zero because the sensor does not measure them. Write $\hat z_t^k=h(\bar\mu_{t,s},\bar\mu_{t,m_k})$ and $\nu_t=z_t-\hat z_t^k$, wrapping the bearing component of the innovation to $[-\pi,\pi)$. For example, bearings just below $\pi$ and just above $-\pi$ are close to one another, not almost a full revolution apart.

The EKF correction is

$$
\begin{aligned}
\Psi_k&=H_t^k\bar\Sigma_tH_t^{k\top}+R_t,\\
K_t&=\bar\Sigma_tH_t^{k\top}\Psi_k^{-1},\\
\mu_t&=\bar\mu_t+K_t\nu_t,\\
\Sigma_t&=\bar\Sigma_t-K_t\Psi_kK_t^\top.
\end{aligned}
$$

The first two equations are familiar from localization. Look more closely at the rows of $K_t$ corresponding to a different landmark $j$:

$$
K_{m_j}=\left(\bar\Sigma_{m_j,s}H_s^\top+\bar\Sigma_{m_j,m_k}H_{m_k}^\top\right)\Psi_k^{-1}.
$$

Although the measurement function does not depend on $m_j$, this gain need not be zero. If it is nonzero, an observation of landmark $k$ can change our estimate of landmark $j$ through the correction $K_{m_j}\nu_t$. We do not need to observe $j$ again in order to refine its position.

When the robot returns to a previously visited area, observations of familiar landmarks can therefore correct portions of the map that are no longer visible. This is the role of loop closure in EKF SLAM. The correction depends on the cross-covariances and on the quality of the association; it is not guaranteed to improve every estimate in a particular realization.

## Unknown correspondences

In practice, the sensor reports a range and bearing but may not identify the landmark. If several pillars look alike, we must first decide which entry in the map corresponds to the observed pillar. We may also be seeing a pillar for the first time.

For an observation $z_t^i$ and a candidate landmark $k$, define the squared Mahalanobis distance

$$
D_{ik}^2=(z_t^i-\hat z_t^k)^\top\Psi_k^{-1}(z_t^i-\hat z_t^k),
$$

again wrapping the bearing residual. In the [correspondence estimation](/aiml-common/lectures/rse/correspondence-estimation/index) section, we derived the maximum likelihood criterion $D_{ik}^2+\ln\det\Psi_k$. The EKF SLAM algorithm in Table 10.2 of Thrun, Burgard and Fox uses just $D_{ik}^2$. The criteria agree when the innovation covariances have the same determinant, and may give different associations otherwise.

The algorithm uses a threshold $\alpha$ to decide whether any existing landmark is a plausible match. If the smallest distance exceeds $\alpha$, the implementation initializes a new landmark. For approximately Gaussian, correctly associated innovations, $D_{ik}^2$ is approximately chi-squared with two degrees of freedom. The implementation uses $\alpha=13.8$, close to the 99.9th percentile. Thus a correct association would fail this gate about once in a thousand measurements under the ideal model. This does not mean that a newly initialized landmark has a 99.9% probability of being real, or that a false association is equally unlikely.

## Adding a landmark

Suppose the measurement comes from a new pillar. Given a range $r$ and bearing $\phi$, we can estimate its position from the current pose by inverting the observation model:

$$
\hat m_{N+1}=\begin{bmatrix}
\bar\mu_{t,x}+r\cos(\phi+\bar\mu_{t,\theta})\\
\bar\mu_{t,y}+r\sin(\phi+\bar\mu_{t,\theta})
\end{bmatrix}.
$$

This construction requires both range and bearing. A monocular bearing-only measurement does not determine depth from a single view; triangulation requires another view with sufficient parallax (or additional information). We return to this in [visual SLAM](/aiml-common/lectures/rse/visual-slam/index).

The new landmark is uncertain for two reasons: the robot pose is uncertain, and the range and bearing are noisy. Both must enter its covariance. Let $G_s$ and $G_z$ be the Jacobians of the placement function above with respect to the pose and measurement. Using first-order uncertainty propagation, assuming the measurement noise is independent of the existing state, we obtain

$$
\begin{aligned}
C&=G_s\bar\Sigma_{s,\cdot},\\
P&=G_s\bar\Sigma_{ss}G_s^\top+G_zR_tG_z^\top.
\end{aligned}
$$

Here $\bar\Sigma_{s,\cdot}$ consists of the three pose rows of the existing covariance. The block $C$ therefore includes the new landmark's cross-covariance with every existing state component. Append the two new coordinates to the mean and augment the covariance:

$$
\Sigma_{\mathrm{aug}}=\begin{bmatrix}\bar\Sigma&C^\top\\C&P\end{bmatrix}.
$$

This is the inverse-measurement initialization used in the implementation, following Corke [@Corke2023-jx]. Thrun, Burgard and Fox instead start each new landmark from a diffuse prior. For subsequent observations in the same time step, use the current estimate and covariance in place of the predicted quantities above.

The first observation of an otherwise unconstrained new landmark cannot tell us where the robot is in the world. Any shift in the robot pose can be accommodated by placing the landmark accordingly. We use this measurement to initialize the landmark; we must not then reuse the same measurement as an independent correction. A later observation provides a constraint between the new robot pose and the stored landmark estimate.

## Limitations of EKF SLAM

The preceding discussion assumes correct landmark associations, stationary landmarks, and a sufficiently accurate linearization. Each assumption has practical consequences.

The pose and map are defined relative to a reference frame. Without an external position or heading reference, translating or rotating the entire trajectory and map together leaves the relative measurements unchanged. We usually choose the initial pose to define the world frame. Loop closure improves the internally estimated geometry, but does not supply an absolute geographical reference.

The EKF approximates a nonlinear posterior by a Gaussian. Large uncertainty, particularly in heading, can make the local linearization inaccurate. Repeated linearization can also make the estimator overconfident in directions that are not observable [@Huang2010-qv]. A small reported covariance is not by itself evidence of a correct map.

A wrong correspondence can cause a large correction to the pose and map. The nearest-neighbor method described here does not revisit earlier association decisions. Practical systems may maintain provisional landmarks, reject ambiguous observations, or remove landmarks that repeatedly fail to be detected. The simple implementation does not use missed detections as measurements. That is a limitation of its observation model, not a general prohibition on incorporating negative information into probabilistic estimation.

Finally, with $N$ landmarks the covariance has $(3+2N)^2$ entries. A single measurement has a two-dimensional innovation, so the correction $K_t\Psi_kK_t^\top$ can be computed as a rank-at-most-two update in $O(N^2)$ time. Computing $(I-K_tH_t)\bar\Sigma_t$ as a general dense matrix product instead can cost $O(N^3)$. A full search through candidate landmarks also adds work. The implementation uses only eight pillars, so clarity of the code is more important than exploiting every available sparsity.

For larger maps, methods such as FastSLAM, submaps, and sparse information filters avoid some of the costs of the dense joint covariance. Graph-based SLAM takes a different approach, retaining pose and measurement constraints and exploiting the sparsity of the resulting optimization problem. See Thrun, Burgard and Fox, Chapters 12 and 13, for the former approaches.

## From theory to implementation

In the [next section](/aiml-common/lectures/rse/ekf-slam/index), we will run EKF SLAM on a TurtleBot in an environment with pillars. The LIDAR and depth camera provide different ways of obtaining range-bearing observations, but the prediction, correction, and landmark initialization equations are the same. As you examine the resulting maps, pay particular attention to what happens when a pillar is observed again after a period of motion.

## Notation reference

| Symbol | Meaning | Size |
| --- | --- | --- |
| $y_t$, $\mu_t$, $\bar\mu_t$ | full state (pose and $N$ landmarks), its mean and predicted mean | $(3 + 2N) \times 1$ |
| $\Sigma_t$, $\bar\Sigma_t$ | covariance of the full state | $(3 + 2N) \times (3 + 2N)$ |
| $s_t$ | robot pose $(x, y, \theta)$ | $3 \times 1$ |
| $a_t$ | wheel speeds | $2 \times 1$ |
| $G_t$ | motion Jacobian with respect to the pose | $3 \times 3$ |
| $V_t$ | motion Jacobian with respect to the wheel speeds | $3 \times 2$ |
| $M_t$ | wheel-speed noise covariance | $2 \times 2$ |
| $Q_t = V_t M_t V_t^\top$ | pose process noise covariance | $3 \times 3$ |
| $z^i_t$, $\hat z^k_t$ | one range-bearing measurement and its prediction for landmark $k$ | $2 \times 1$ |
| $H^k_t$ | measurement Jacobian for landmark $k$ | $2 \times (3 + 2N)$ |
| $R_t$ | measurement noise covariance | $2 \times 2$ |
| $\Psi_k$ | innovation covariance | $2 \times 2$ |
| $K_t = \bar\Sigma_t H^{k\top}_t \Psi_k^{-1}$ | Kalman gain for one measurement | $(3 + 2N) \times 2$ |
| $G_s$, $G_z$ | placement Jacobians with respect to the pose and the measurement | $2 \times 3$, $2 \times 2$ |
| $\Sigma_{m_{N+1}}$ | covariance of a new landmark | $2 \times 2$ |
| $\Sigma_{m_{N+1}, \cdot}$ | its cross-covariance with the existing state | $2 \times (3 + 2N)$ |

## Lectures

<iframe
  width="100%"
  height="400"
  src="https://www.youtube.com/embed/0I30M6yTklo"
  title="SLAM Overview"
  frameBorder="0"
  allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture"
  allowFullScreen
/>

<iframe
  width="100%"
  height="400"
  src="https://www.youtube.com/embed/DE6Jn2cB4J4"
  title="SLAM Course 04 - Extended Kalman Filter (Cyrill Stachniss)"
  frameBorder="0"
  allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture"
  allowFullScreen
/>

<iframe
  width="100%"
  height="400"
  src="https://www.youtube.com/embed/XeWG5D71gC0"
  title="SLAM Course 05 - EKF SLAM (Cyrill Stachniss)"
  frameBorder="0"
  allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture"
  allowFullScreen
/>

Key references: [@Thrun2005-dx; @Corke2023-jx; @Dissanayake2001-ix; @Huang2010-qv]